# 04 · DuckDB SQL Analytics over Parquet Marts

This is the *lakehouse* part of the lab. The build step exports Parquet marts; here we query those files directly with DuckDB SQL — no server, no Python transforms, just an in-memory engine reading Parquet.

This is exactly how you would explore the real OWID marts at scale.

## Setup

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upwards until the directory containing pyproject.toml is found."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return here


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "raw"
DB_PATH = REPO_ROOT / "data" / "processed" / "carbon_transition.duckdb"
MARTS_DIR = REPO_ROOT / "data" / "processed" / "marts"
REPORTS_DIR = REPO_ROOT / "reports" / "sample_run"
print(f"Repository root: {REPO_ROOT}")


Repository root: C:\Users\diogo\work_code\portfolio\carbon-transition-duckdb-lab


In [2]:
from carbon_transition_duckdb.config import ProjectPaths
from carbon_transition_duckdb.pipeline import build_duckdb_lakehouse
from carbon_transition_duckdb.sample_data import generate_synthetic_owid_data

# Build the lakehouse on demand so every notebook is runnable in isolation.
if not DB_PATH.exists():
    generate_synthetic_owid_data(RAW_DIR, start_year=2010, end_year=2024)
    build_duckdb_lakehouse(
        ProjectPaths(raw_dir=RAW_DIR, database=DB_PATH, export_dir=MARTS_DIR)
    )
    print("Built DuckDB lakehouse.")
else:
    print("Reusing existing DuckDB lakehouse.")


Reusing existing DuckDB lakehouse.


In [3]:
import duckdb

mart_file = (MARTS_DIR / 'mart_country_year_transition.parquet').as_posix()
# A fresh in-memory database that reads the Parquet file lazily.
con = duckdb.connect()
con.execute(f"CREATE VIEW mart AS SELECT * FROM read_parquet('{mart_file}')")
con.execute('SELECT COUNT(*) AS rows, COUNT(DISTINCT country) AS countries FROM mart').fetchdf()

,rows,countries
0,75,5


## 1. Highest emitters in the latest year

A correlated subquery pins the query to the most recent year in the mart.

In [4]:
con.execute(
    """
    SELECT country, year, co2, co2_per_capita, fossil_share_energy
    FROM mart
    WHERE year = (SELECT MAX(year) FROM mart)
    ORDER BY co2 DESC
    """
).fetchdf()

,country,year,co2,co2_per_capita,fossil_share_energy
0,Deltora,2024,254.4571,7.2898,83.9179
1,Borealia,2024,157.3759,7.7647,75.1255
2,Atlantis,2024,52.6251,5.8420,61.1859
3,Estavia,2024,27.0308,2.0005,30.8146
4,Cyrenia,2024,9.1096,1.7978,21.3457


## 2. Year-over-year change with window functions

`LAG` over a partition computes per-country change without any Python loop.

In [5]:
yoy = con.execute(
    """
    SELECT
        country,
        year,
        co2,
        co2 - LAG(co2) OVER (PARTITION BY country ORDER BY year) AS co2_yoy_change,
        ROUND(
            100.0 * (co2 - LAG(co2) OVER (PARTITION BY country ORDER BY year))
            / NULLIF(LAG(co2) OVER (PARTITION BY country ORDER BY year), 0), 2
        ) AS co2_yoy_pct
    FROM mart
    QUALIFY year >= (SELECT MAX(year) - 4 FROM mart)
    ORDER BY country, year
    """
).fetchdf()
yoy

,country,year,co2,co2_yoy_change,co2_yoy_pct
0,Atlantis,2020,55.4456,2.9591,5.64
1,Atlantis,2021,51.3215,-4.1241,-7.44
2,Atlantis,2022,49.6705,-1.6510,-3.22
3,Atlantis,2023,51.6290,1.9585,3.94
4,Atlantis,2024,52.6251,0.9961,1.93
5,Borealia,2020,163.8342,-0.6888,-0.42
6,Borealia,2021,169.8864,6.0522,3.69
7,Borealia,2022,172.1958,2.3094,1.36
8,Borealia,2023,170.4533,-1.7425,-1.01
9,Borealia,2024,157.3759,-13.0774,-7.67


## 3. Multi-year emissions CAGR

Compound annual growth rate of CO2 over the full panel, per country — a single SQL statement combining `FIRST`/`LAST` ordered aggregates.

In [6]:
cagr = con.execute(
    """
    WITH bounds AS (
        SELECT
            country,
            arg_min(co2, year) AS first_co2,
            arg_max(co2, year) AS last_co2,
            MAX(year) - MIN(year) AS span_years
        FROM mart
        GROUP BY country
    )
    SELECT
        country,
        ROUND(first_co2, 1) AS first_co2,
        ROUND(last_co2, 1) AS last_co2,
        span_years,
        ROUND(100.0 * (POWER(last_co2 / first_co2, 1.0 / span_years) - 1), 2) AS co2_cagr_pct
    FROM bounds
    ORDER BY co2_cagr_pct DESC
    """
).fetchdf()
cagr

,country,first_co2,last_co2,span_years,co2_cagr_pct
0,Deltora,260.1,254.5,14,-0.16
1,Borealia,175.7,157.4,14,-0.78
2,Atlantis,60.5,52.6,14,-0.99
3,Estavia,42.2,27.0,14,-3.14
4,Cyrenia,19.1,9.1,14,-5.16


## 4. Rank countries within each year

`RANK` over a per-year partition shows whether the fossil-dependence pecking order is stable across the panel.

In [7]:
ranked = con.execute(
    """
    SELECT
        year,
        country,
        ROUND(fossil_share_energy, 1) AS fossil_share_energy,
        RANK() OVER (PARTITION BY year ORDER BY fossil_share_energy DESC) AS fossil_rank
    FROM mart
    WHERE year IN (SELECT MIN(year) FROM mart UNION SELECT MAX(year) FROM mart)
    ORDER BY year, fossil_rank
    """
).fetchdf()
ranked

,year,country,fossil_share_energy,fossil_rank
0,2010,Deltora,87.8,1
1,2010,Borealia,82.1,2
2,2010,Atlantis,74.9,3
3,2010,Estavia,52.3,4
4,2010,Cyrenia,44.7,5
5,2024,Deltora,83.9,1
6,2024,Borealia,75.1,2
7,2024,Atlantis,61.2,3
8,2024,Estavia,30.8,4
9,2024,Cyrenia,21.3,5


## 5. Export a custom mart back to Parquet

Persist a derived table straight from SQL — the same `COPY ... TO` pattern the build step uses.

In [8]:
custom_path = (MARTS_DIR / 'mart_co2_cagr.parquet').as_posix()
con.execute(
    f"""
    COPY (
        WITH bounds AS (
            SELECT country, arg_min(co2, year) AS first_co2,
                   arg_max(co2, year) AS last_co2, MAX(year) - MIN(year) AS span_years
            FROM mart GROUP BY country
        )
        SELECT country,
               ROUND(100.0 * (POWER(last_co2 / first_co2, 1.0 / span_years) - 1), 2) AS co2_cagr_pct
        FROM bounds
    ) TO '{custom_path}' (FORMAT PARQUET)
    """
)
# Read it straight back to confirm the round-trip.
read_back = f"SELECT * FROM read_parquet('{custom_path}') ORDER BY co2_cagr_pct DESC"
check = con.execute(read_back).fetchdf()
con.close()
print('Wrote', Path(custom_path).name)
check

Wrote mart_co2_cagr.parquet


,country,co2_cagr_pct
0,Deltora,-0.16
1,Borealia,-0.78
2,Atlantis,-0.99
3,Estavia,-3.14
4,Cyrenia,-5.16


## Takeaways

- DuckDB reads Parquet marts directly — the marts *are* the API, queryable from any tool that speaks SQL.
- Window functions (`LAG`, `RANK`, `QUALIFY`) and ordered aggregates (`arg_min`/`arg_max`) keep multi-year analytics in one declarative statement.
- `COPY ... TO` lets a notebook publish a new mart without leaving SQL — the foundation for a dbt-style modelling layer (see the roadmap).